# Tahap 2 - Case Representation

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")

print("Jumlah data inventory:", len(df_inventory))
print("Jumlah file txt di data/raw:", len(list(RAW_DIR.glob("*.txt"))))

df_inventory[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "raw_file",
    "status_download",
    "jumlah_kata"
]].head(40)

Jumlah data inventory: 40
Jumlah file txt di data/raw: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,case_001.txt,berhasil_list_html,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,case_002.txt,berhasil_list_html,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,case_003.txt,berhasil_list_html,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,case_004.txt,berhasil_list_html,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,case_005.txt,berhasil_list_html,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,case_006.txt,berhasil_list_html,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,case_007.txt,berhasil_list_html,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,case_008.txt,berhasil_list_html,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,case_009.txt,berhasil_list_html,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,case_010.txt,berhasil_list_html,335


In [2]:
from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"
cases_path = PROCESSED_DIR / "cases.csv"

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")


def normalize_text(text):
    text = str(text)
    text = re.sub(r"\r", " ", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_between(text, start_label, end_labels):
    text = normalize_text(text)
    
    end_pattern = "|".join([re.escape(label) for label in end_labels])
    
    pattern = rf"{re.escape(start_label)}\s*(.*?)(?=\s*(?:{end_pattern})\s*|$)"
    
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    
    if match:
        return normalize_text(match.group(1))
    
    return ""


def extract_penuntut_umum(text):
    text = normalize_text(text)
    
    patterns = [
        r"Penuntut\s+Umum\s*:?\s*(.*?)(?=Terdakwa|Nomor|Tingkat Proses|Klasifikasi|$)",
        r"Penuntut\s+Umum\s*(.*?)(?=Terdakwa|Nomor|Tingkat Proses|Klasifikasi|$)"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            return normalize_text(match.group(1))[:200]
    
    return ""


def extract_terdakwa(text):
    text = normalize_text(text)
    
    patterns = [
        r"Terdakwa\s*:?\s*(.*?)(?=Nomor|Tingkat Proses|Klasifikasi|Kata Kunci|Tahun|Hakim|Amar|$)",
        r"—\s*(.*?)(?=Nomor|Tingkat Proses|Klasifikasi|Kata Kunci|Tahun|Hakim|Amar|$)"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            hasil = normalize_text(match.group(1))
            hasil = hasil.replace("Penuntut Umum:", "")
            return hasil[:300]
    
    return ""


def extract_pasal(text):
    text = normalize_text(text)
    
    patterns = [
        r"Pasal\s+[0-9]+[A-Za-z]?(?:\s+ayat\s+\([0-9]+\))?(?:\s+ke[-\s]?[0-9]+)?(?:\s+KUHP)?",
        r"Pasal\s+[0-9]+[A-Za-z]?\s+KUHP",
        r"Pasal\s+[0-9]+[A-Za-z]?"
    ]
    
    hasil = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for m in matches:
            m = normalize_text(m)
            if m not in hasil:
                hasil.append(m)
    
    return "; ".join(hasil[:10])


def extract_putusan_label(text):
    text_lower = text.lower()
    
    if "pidana penjara" in text_lower:
        return "Pidana Penjara"
    if "hukum" in text_lower and "bulan" in text_lower:
        return "Pidana Penjara"
    if "tahun" in text_lower and "bulan" in text_lower:
        return "Pidana Penjara"
    if "bebas" in text_lower:
        return "Bebas"
    if "lepas" in text_lower:
        return "Lepas"
    
    return "Lain-lain"


def extract_lama_pidana(text):
    text = normalize_text(text)
    
    patterns = [
        r"selama\s+([0-9]+)\s*\([^)]+\)\s*tahun(?:,\s*)?\s*([0-9]+)?\s*(?:\([^)]+\))?\s*bulan?",
        r"selama\s+([0-9]+)\s*\([^)]+\)\s*bulan",
        r"HUKUM\s+([0-9]+)\s+TAHUN\s+([0-9]+)\s+BULAN",
        r"HUKUM\s+([0-9]+)\s+BULAN"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return normalize_text(match.group(0))
    
    return ""


cases = []

for _, row in df_inventory.iterrows():
    case_id = row.get("case_id", "")
    raw_file = row.get("raw_file", "")
    raw_path = RAW_DIR / raw_file
    
    if not raw_path.exists():
        print(f"File tidak ditemukan: {raw_file}")
        continue
    
    text = raw_path.read_text(encoding="utf-8", errors="ignore")
    text = normalize_text(text)
    
    jumlah_kata = len(text.split())
    
    penuntut_umum = extract_penuntut_umum(text)
    terdakwa = extract_terdakwa(text)
    pasal = extract_pasal(text)
    
    hakim_ketua = extract_between(text, "Hakim Ketua", ["Hakim Anggota", "Panitera", "Amar"])
    hakim_anggota = extract_between(text, "Hakim Anggota", ["Panitera", "Amar", "Amar Lainnya"])
    panitera = extract_between(text, "Panitera", ["Amar", "Amar Lainnya", "Catatan Amar"])
    
    amar = extract_between(text, "Amar", ["Amar Lainnya", "Catatan Amar", "Tanggal Musyawarah"])
    amar_lainnya = extract_between(text, "Amar Lainnya", ["Catatan Amar", "Tanggal Musyawarah", "Tanggal Dibacakan"])
    catatan_amar = extract_between(text, "Catatan Amar", ["Tanggal Musyawarah", "Tanggal Dibacakan", "Kaidah", "Abstrak"])
    
    tanggal_musyawarah = extract_between(text, "Tanggal Musyawarah", ["Tanggal Dibacakan", "Kaidah", "Abstrak"])
    tanggal_dibacakan = extract_between(text, "Tanggal Dibacakan", ["Kaidah", "Abstrak", "Lampiran"])
    
    if catatan_amar:
        ringkasan_fakta = catatan_amar
    else:
        ringkasan_fakta = text[:1200]
    
    if catatan_amar:
        argumen_hukum = catatan_amar
    else:
        argumen_hukum = amar_lainnya
    
    solution_text = catatan_amar if catatan_amar else amar_lainnya
    solution_label = extract_putusan_label(solution_text + " " + amar_lainnya)
    lama_pidana = extract_lama_pidana(solution_text + " " + amar_lainnya)
    
    cases.append({
        "case_id": case_id,
        "no_perkara": row.get("no_perkara", ""),
        "tanggal_putusan": row.get("tanggal_putusan", ""),
        "pengadilan": row.get("pengadilan", "PN Tangerang"),
        "jenis_perkara": row.get("jenis_perkara", "Pidana Umum - Pencurian"),
        "penuntut_umum": penuntut_umum,
        "terdakwa": terdakwa,
        "pasal": pasal,
        "hakim_ketua": hakim_ketua,
        "hakim_anggota": hakim_anggota,
        "panitera": panitera,
        "amar": amar,
        "amar_lainnya": amar_lainnya,
        "catatan_amar": catatan_amar,
        "tanggal_musyawarah": tanggal_musyawarah,
        "tanggal_dibacakan": tanggal_dibacakan,
        "ringkasan_fakta": ringkasan_fakta,
        "argumen_hukum": argumen_hukum,
        "solution_text": solution_text,
        "solution_label": solution_label,
        "lama_pidana": lama_pidana,
        "sumber_url": row.get("sumber_url", ""),
        "raw_file": raw_file,
        "jumlah_kata": jumlah_kata,
        "text_full": text
    })

cases_df = pd.DataFrame(cases)

cases_df.to_csv(cases_path, index=False)

print("cases.csv berhasil dibuat.")
print("Lokasi:", cases_path)
print("Jumlah kasus:", len(cases_df))

cases_df.head()

cases.csv berhasil dibuat.
Lokasi: /home/zack/Penalaran-Komputer-subcpmk-3/data/processed/cases.csv
Jumlah kasus: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,penuntut_umum,terdakwa,pasal,hakim_ketua,hakim_anggota,...,tanggal_dibacakan,ringkasan_fakta,argumen_hukum,solution_text,solution_label,lama_pidana,sumber_url,raw_file,jumlah_kata,text_full
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,,JAMALUDIN Bin SANIF terbukti secara sah danmey...,Pasal 363 ayat (2) KUHP; Pasal 363,,,...,,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...,,,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,80,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,;Menjatuhkan pidana penjara terhadap,AMIN Als. UBE Bin UDIN danALIP KURNIAWAN Als. ...,Pasal 365 ayat (2) ke2 KUHP; Pasal 365,,,...,,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,322,Pengadilan PN TANGERANG Pidana Umum Pencurian ...
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,,,...,,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...,,,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,123,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,"REZA VAHLEVI, SH",MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN 37 ...,Pasal 362 KUHP; Pasal 362,,,...,,Pengadilan PN TANGERANG Pidana Umum Register :...,,,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,213,Pengadilan PN TANGERANG Pidana Umum Register :...
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,di mukapersidangan ;Telah mendengar keterangan...,di persidangan ;Telah mendengar pembacaan tunt...,Pasal 362 KUHP; Pasal 362,,,...,,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,342,Pengadilan PN TANGERANG Pidana Umum Pencurian ...


In [3]:
cases_df = pd.read_csv(cases_path, dtype=str).fillna("")

print("Jumlah kasus:", len(cases_df))
print("Duplikat case_id:", cases_df["case_id"].duplicated().sum())
print("Duplikat no_perkara:", cases_df["no_perkara"].duplicated().sum())

print("\nJumlah kata:")
print(cases_df["jumlah_kata"].astype(int).describe())

print("\nDistribusi solution_label:")
print(cases_df["solution_label"].value_counts())

cases_df[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "terdakwa",
    "amar_lainnya",
    "solution_label",
    "lama_pidana",
    "jumlah_kata"
]].head(40)

Jumlah kasus: 40
Duplikat case_id: 0
Duplikat no_perkara: 0

Jumlah kata:
count     40.000000
mean     193.875000
std      116.856736
min       58.000000
25%       97.000000
50%      115.500000
75%      323.750000
max      373.000000
Name: jumlah_kata, dtype: float64

Distribusi solution_label:
solution_label
Lain-lain    40
Name: count, dtype: int64


,case_id,no_perkara,tanggal_putusan,terdakwa,amar_lainnya,solution_label,lama_pidana,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,JAMALUDIN Bin SANIF terbukti secara sah danmey...,,Lain-lain,,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,AMIN Als. UBE Bin UDIN danALIP KURNIAWAN Als. ...,,Lain-lain,,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,Lain-lain,,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN 37 ...,,Lain-lain,,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,di persidangan ;Telah mendengar pembacaan tunt...,,Lain-lain,,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,"NUR APRIYANI Binti (Alm) NURDIN, terbuktibersa...",,Lain-lain,,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,DEDY KUMALA BIN SAHLAN bersalah secara syahdan...,,Lain-lain,,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,YUDI HARYADI Bin BAROZI telah bersalahmelakuka...,,Lain-lain,,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,1.AFRIYAZI Als AJI Bin EKO EDI PRIYANTO 2.WAHY...,,Lain-lain,,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,1.ICON SIREGAR Alias ICON Bin SAMSIR ALAM SIRE...,,Lain-lain,,335
